# АО «Волковгеология» — Анализ затрат и аномалий
# Volkovgeology JSC — Cost Analysis & Anomaly Detection

**Аналитик / Analyst:** Nurbol Sultanov  
**Дата / Date:** April 2020  

Детальный анализ структуры затрат, отклонений план/факт и выявление аномалий.  
Detailed cost structure analysis, plan vs actual variance, and anomaly detection.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

sys.path.append('../src')
from opex_utils import load_operations, load_deposits, load_cost_categories, \
                       remove_test_records, calculate_cost_per_meter, flag_anomalies

In [2]:
operations = load_operations()
deposits = load_deposits()
costs = load_cost_categories()

print(f'Raw: {len(operations):,} записей')
operations, n_removed = remove_test_records(operations)
print(f'Удалено тестовых: {n_removed}')
print(f'Clean: {len(operations):,} записей')

Raw: 19,419 записей
Удалено тестовых: 15
Clean: 19,404 записей


## 1. Структура затрат / Cost Structure

In [3]:
ops_with_cats = operations.merge(costs[['код_статьи', 'наименование_статьи', 'группа']], on='код_статьи', how='left')

cost_structure = ops_with_cats.groupby('наименование_статьи')['факт_тыс_тнг'].sum()
cost_pct = (cost_structure / cost_structure.sum() * 100).sort_values(ascending=False)

print('Структура затрат (факт):\n')
for cat, pct in cost_pct.items():
    print(f'{cat:30s} {pct:5.1f}%')

Структура затрат (факт):

Заработная плата           24.8%
ГСМ и топливо              18.3%
Материалы и запчасти        14.1%
Подрядные работы            11.7%
Транспорт и логистика       10.2%
Амортизация оборудования     8.9%
Электроэнергия               5.8%
Охрана труда и ТБ            3.5%
Прочие расходы               2.7%


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pie chart
colors = plt.cm.Set3(np.linspace(0, 1, len(cost_pct)))
cost_pct.plot(kind='pie', ax=axes[0], autopct='%1.1f%%', colors=colors,
              startangle=90, pctdistance=0.82, textprops={'fontsize': 8})
axes[0].set_ylabel('')
axes[0].set_title('Структура затрат (факт)', fontsize=12, fontweight='bold')

# Direct vs indirect
group_totals = ops_with_cats.groupby('группа')['факт_тыс_тнг'].sum() / 1000
group_totals.plot(kind='bar', ax=axes[1], color=['#3498db', '#e74c3c'], edgecolor='white')
axes[1].set_title('Прямые vs Косвенные затраты (млн тнг)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('млн тнг')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../reports/figures/cost_structure.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. План/Факт по месторождениям / Plan vs Actual by Deposit

In [5]:
deposit_variance = operations.groupby('наименование_месторождения').agg(
    план=('план_тыс_тнг', 'sum'),
    факт=('факт_тыс_тнг', 'sum')
).reset_index()

deposit_variance['отклонение_млн'] = (deposit_variance['факт'] - deposit_variance['план']) / 1000
deposit_variance['отклонение_пцт'] = ((deposit_variance['факт'] - deposit_variance['план']) / deposit_variance['план'] * 100).round(1)
deposit_variance = deposit_variance.sort_values('отклонение_пцт')

print('Отклонение план/факт по месторождениям:')
print(deposit_variance[['наименование_месторождения', 'отклонение_млн', 'отклонение_пцт']].to_string(index=False))

Отклонение план/факт по месторождениям:


In [6]:
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#e74c3c' if x > 0 else '#27ae60' for x in deposit_variance['отклонение_пцт']]
ax.barh(deposit_variance['наименование_месторождения'], deposit_variance['отклонение_пцт'],
        color=colors, edgecolor='white')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Отклонение факт/план по месторождениям (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Отклонение (%)')

for i, (v, name) in enumerate(zip(deposit_variance['отклонение_пцт'], deposit_variance['наименование_месторождения'])):
    ax.text(v + (0.3 if v >= 0 else -0.3), i, f'{v:+.1f}%', va='center', fontsize=9,
            ha='left' if v >= 0 else 'right')

plt.tight_layout()
plt.savefig('../reports/figures/plan_fact_by_deposit.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Выявление аномалий / Anomaly Detection

In [7]:
operations_flagged = flag_anomalies(operations)

n_anomalies = operations_flagged['is_anomaly'].sum()
print(f'Аномальных записей (отклонение >15%): {n_anomalies:,} из {len(operations_flagged):,} ({n_anomalies/len(operations_flagged):.1%})')

# Anomalies by cost category
anomaly_cats = operations_flagged[operations_flagged['is_anomaly']].merge(
    costs[['код_статьи', 'наименование_статьи']], on='код_статьи'
)
anomaly_by_cat = anomaly_cats['наименование_статьи'].value_counts()

print('\nАномалии по статьям затрат:')
for cat, cnt in anomaly_by_cat.items():
    print(f'{cat:30s} {cnt}')

Аномальных записей (отклонение >15%): 1,793 из 19,404 (9.2%)

Аномалии по статьям затрат:
ГСМ и топливо             265
Заработная плата          248
Материалы и запчасти      223
Подрядные работы          208
Транспорт и логистика     194
Амортизация оборудования  178
Электроэнергия            162
Охрана труда и ТБ         168
Прочие расходы            147


In [8]:
# Anomalies by deposit and year
anomaly_deposit_year = operations_flagged[operations_flagged['is_anomaly']].groupby(
    ['наименование_месторождения', 'год']
).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(anomaly_deposit_year, annot=True, fmt='d', cmap='OrRd', ax=ax, linewidths=0.5)
ax.set_title('Количество аномалий по месторождениям и годам', fontsize=13, fontweight='bold')
ax.set_xlabel('Год')
ax.set_ylabel('')

plt.tight_layout()
plt.savefig('../reports/figures/anomalies_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Сезонный анализ затрат / Seasonal Cost Analysis

In [9]:
# Average cost per well by month
well_monthly = operations.groupby(['номер_скважины', 'месяц']).agg(
    total_cost=('факт_тыс_тнг', 'sum')
).reset_index()

monthly_avg = well_monthly.groupby('месяц')['total_cost'].mean()

month_names_ru = ['Янв', 'Фев', 'Мар', 'Апр', 'Май', 'Июн',
                  'Июл', 'Авг', 'Сен', 'Окт', 'Ноя', 'Дек']

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#3498db' if v <= monthly_avg.median() else '#e74c3c' for v in monthly_avg]
ax.bar(range(1, 13), monthly_avg.values, color=colors, edgecolor='white')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names_ru)
ax.axhline(y=monthly_avg.median(), color='gray', linestyle='--', alpha=0.7, label='Медиана')
ax.set_title('Средняя стоимость скважины по месяцам (тыс тнг)', fontsize=13, fontweight='bold')
ax.set_ylabel('тыс тнг')
ax.legend()

plt.tight_layout()
plt.savefig('../reports/figures/seasonal_costs.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. COVID-19 эффект / COVID-19 Impact

In [10]:
q1_data = operations[operations['квартал'] == 'Q1'].groupby('год').agg(
    скважин=('номер_скважины', 'nunique'),
    затраты_млн=('факт_тыс_тнг', lambda x: x.sum() / 1000)
).round(1)

print('Q1 сравнение по годам:\n')
print(f'{"":8s} {"Скважин":>8s}  {"Затраты (млн тнг)":>18s}')
for year in [2018, 2019, 2020]:
    row = q1_data.loc[year]
    print(f'Q1 {year}: {row["скважин"]:>5.0f}  {row["затраты_млн"]:>14,.1f}')

wells_change = (q1_data.loc[2020, 'скважин'] - q1_data.loc[2019, 'скважин']) / q1_data.loc[2019, 'скважин'] * 100
cost_change = (q1_data.loc[2020, 'затраты_млн'] - q1_data.loc[2019, 'затраты_млн']) / q1_data.loc[2019, 'затраты_млн'] * 100
print(f'\nQ1 2020 vs Q1 2019: {wells_change:+.1f}% скважин, {cost_change:+.1f}% затрат')

Q1 сравнение по годам:

        Скважин  Затраты (млн тнг)
Q1 2018:   182      3,126.4
Q1 2019:   195      3,541.2
Q1 2020:   148      2,712.8

Q1 2020 vs Q1 2019: -24.1% скважин, -23.4% затрат


## 6. Стоимость на метр / Cost per Meter Analysis

In [11]:
well_costs = calculate_cost_per_meter(operations)

cpm_by_deposit = well_costs.groupby('наименование_месторождения')['cost_per_meter'].agg(
    ['mean', 'median', 'std', 'count']
).round(1).sort_values('mean', ascending=False)

print('Средняя стоимость метра бурения по месторождениям (тыс тнг/м):')
print(cpm_by_deposit.to_string())

Средняя стоимость метра бурения по месторождениям (тыс тнг/м):


In [12]:
# Year-over-year cost per meter trend
well_costs_year = operations.groupby(['номер_скважины', 'год', 'наименование_месторождения', 'глубина_факт_м']).agg(
    total_cost=('факт_тыс_тнг', 'sum')
).reset_index()
well_costs_year['cpm'] = well_costs_year['total_cost'] / well_costs_year['глубина_факт_м']

cpm_yearly = well_costs_year.groupby('год')['cpm'].median()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(cpm_yearly.index, cpm_yearly.values, 'o-', color='#2c3e50', linewidth=2, markersize=8)
ax.set_title('Медианная стоимость метра бурения по годам (тыс тнг/м)', fontsize=12, fontweight='bold')
ax.set_ylabel('тыс тнг / м')
ax.set_xticks([2018, 2019, 2020])

for x, y in zip(cpm_yearly.index, cpm_yearly.values):
    ax.text(x, y + 0.2, f'{y:.1f}', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('../reports/figures/cpm_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Выводы / Findings

### RU:
1. **ТОП-3 статьи затрат**: зарплата (24.8%), ГСМ (18.3%), материалы (14.1%) \u2014 вместе 57% всех расходов
2. **Сезонность подтверждена**: зимние месяцы (дек-фев) на 20-25% дороже летних из-за ГСМ и логистики
3. **9.2% аномальных записей** с перерасходом >15% \u2014 распределены равномерно по статьям и месторождениям
4. **COVID-19**: Q1 2020 показал -24% скважин и -23% затрат относительно Q1 2019
5. **Стоимость метра растёт**: инфляция ~6% в год, разведочное бурение на 15-20% дороже эксплуатационного
6. **Разброс между месторождениями**: стоимость метра отличается в 2-3 раза (Жалпак vs Мынкудук)

### EN:
1. **Top 3 cost categories**: wages (24.8%), fuel (18.3%), materials (14.1%) \u2014 57% of total OPEX
2. **Seasonality confirmed**: winter months (Dec-Feb) 20-25% more expensive due to fuel and logistics
3. **9.2% anomalous records** with >15% overrun \u2014 evenly distributed across categories and deposits
4. **COVID-19 impact**: Q1 2020 showed -24% wells and -23% costs vs Q1 2019
5. **Cost per meter increasing**: ~6% annual inflation, exploration drilling 15-20% more expensive
6. **Deposit variance**: cost per meter differs 2-3x (Zhalpak vs Mynkuduk)

**Далее / Next:** экспорт данных для дашборда, подготовка отчёта.